# WRF Workflow Info
## Setup

### Modules
Modules are a method for making software built for Derecho and Casper available from your shell. They have to be loaded each time you login so I load them in my bashrc. Here is what I was using.

```bash
set -o vi

module load conda
module load neovim
module load nco
module load ncl
module load gdal
module load cdo
module load wgrib2

alias qburn='qhist -u ljaeger -d 50 | awk '"'"'NR>2 { sum += $NF } END { print sum * 128 * 1.5 - 300000}'"'"''
alias qrn='qstat -u ljaeger | awk '"'"'NR>2 && $(NF-1)=="R" { n++ } END { print n * 128 * 1.5 }'"'"''

conda activate workflow

export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && \. "$NVM_DIR/nvm.sh"  
[ -s "$NVM_DIR/bash_completion" ] && \. "$NVM_DIR/bash_completion" bash_completion
```

### Conda
The conda files to reproduce the env I am using are at environment.yml and environment.exact.yml. The second just has more precise version pinning. 

### Binaries
/glade/u/home/ljaeger/wrf-run/bin

You'll need WPS-4.6-dmpar and WRF-4.6

## Preprocessing

For each day long simulation we need a few config files and jobs submission scripts. The preprocessing step is just taking template versions of these configs and changing out fire ids, dates, and coordinates.


In [ ]:
# combine yaml files from wsts and turn into a single csv
!python fires/combine_fire_yamls.py

# remove fires randomly until under budget and output to another csv
!fires/filter_fires_by_budget.py

# generate the actual configs from from the previous csv
!python fires/generate_configs.py \
    --runs-csv fires/runs.csv \
    --workflow-template configs/templates/workflow/base.yaml \
    --wrf-template-dir configs/templates/wrf \
    --output-dir configs/built \
    --workflow-root /glade/derecho/scratch/ljaeger/workflow \
    --grib-root /glade/derecho/scratch/ljaeger/data/hrrr \
    --overwrite

Input CSV fires/runs.csv does not exist.


## Runs